# 🐯 TIGeR — Corrected Detection Run (synthetic catalogue)

Regenerates the **detection** table: precision / recall / F1 and per-error-type
recall on the seeded synthetic catalogue.

**Scope changed.** This notebook previously also ran the Kaggle Myntra fashion
set. Both real-data verticals now come from official ABO — see
`tiger_abo_corrected_run.ipynb`. What remains here is the synthetic benchmark,
which is where the project's strongest verified result lives: a
CLIP-similarity-only detector catches **0.267** of text mutations, and the
per-field contrastive probes lift that to **0.853**.

That result is independent of the dataset migration, so this notebook is
unaffected by it — but it does need re-running, because `synthgen` has been
unbuildable since the ABO schema edit (D5) and the per-signal precision floor
now scores each probe against the error it names rather than any dirty row (A8).

**What was wrong with the harness**

| # | Defect | Effect |
|---|---|---|
| A8 | Per-signal precision counted any dirty row as a hit | The 0.85 floor certified "fires on dirty rows", not "identifies the error it names". Expect some probes to be quarantined now that were not. |
| A7 | Fusion was never loaded by `detect` | The advertised P=0.888 was an offline ablation row; the live pipeline ran at P=0.793. Now opt-in via `--fusion`. |
| D5 | Five colours were both domain values and aliases | Probes scored duplicates as rivals, and `synthgen` crashed on ~1/3 of products. |

**Runtime:** ~15–25 min on a T4. *Notebook options → Accelerator → GPU T4 x2.*

## Step 0 · Clone the fixed branch and install

In [ ]:
!rm -rf TIGeR-Text-Image-Generative-Repair
!git clone -q -b docs/fixes-backlog-audit https://github.com/namaray/TIGeR-Text-Image-Generative-Repair.git
%cd TIGeR-Text-Image-Generative-Repair
!git log --oneline -1

# H7: never use --force-reinstall here. It rebuilds the whole dependency tree,
# pulls an incompatible torchvision, and every later cell dies with
# "operator torchvision::nms does not exist". Clear only the local package.
!pip uninstall -y tiger -q 2>/dev/null
!pip install --no-cache-dir -e ".[dev,vlm,gen]" -q
print("\n✅ installed")

## Step 1 · Guard: confirm the fixes are present

Cheap, and it fails loudly. Cloning the wrong branch otherwise wastes the run
and produces numbers that look plausible and are not.

In [ ]:
import inspect
from tiger import cli
from tiger.schema import Schema, load_schema
from tiger.fusion import PROBE_TARGETS
from tiger.eval import repair_ablation as RA
from pathlib import Path

checks = []
def check(name, ok, detail=""):
    checks.append((name, bool(ok), detail))

src = inspect.getsource(RA.run_repair_ablations)
check("A1  gamma written to cfg['arbiter']",
      'cfg_no_gamma.setdefault("arbiter", {})["gamma"] = 0.0' in src)
check("A4  configs deep-copied", "copy.deepcopy(cfg)" in src)
check("A3  baseline emits no E4 key",
      '"E4"' not in inspect.getsource(RA.DummyArbiter.predict_proba))
check("A3b baseline is seeded",
      "seed" in inspect.signature(RA.DummyArbiter.__init__).parameters)
check("A5  every audited field scored", hasattr(RA, "truth_from_audit"))
check("A5  image repairs scored", hasattr(RA, "image_provenance"))
check("A7  --fusion flag exposed", '"--fusion"' in inspect.getsource(cli.main))
check("A8  probes scored on-target", isinstance(PROBE_TARGETS, dict) and "color" in PROBE_TARGETS)
check("D5  surface_forms available", hasattr(Schema, "surface_forms"))

# --- Phase 1: the ABO vertical migration ---
from tiger.data.import_abo import CATEGORY_MAP, VERTICALS
from tiger.data import abo_vocab

check("P1  two ABO verticals defined",
      set(VERTICALS) == {"furnishing", "accessories"} and len(CATEGORY_MAP) == 15,
      f"{len(CATEGORY_MAP)} product types")
check("P1  phone cases excluded", "CELLULAR_PHONE_CASE" not in CATEGORY_MAP)
check("P1  colour normaliser present", hasattr(abo_vocab, "normalize_color"))
check("P1  legacy config.yaml gone", not Path("configs/config.yaml").exists())

schema = load_schema("configs/schema.yaml")
dom = schema.domain("color")
check("D5  no duplicate colours in the domain",
      len(dom) == len({schema.normalize("color", v) for v in dom}),
      f"{len(dom)} entries: {dom}")

from tiger.data.synthgen import COLOR_RGB
missing = [c for c in dom if c != "multicolour" and c not in COLOR_RGB]
check("D5  synthgen can render every colour", not missing, f"missing: {missing}")

cfg = cli.load_cfg()
allowed = cfg["arbiter"]["t2v_policy"]["allowed_categories"]
missing_t2v = sorted(set(CATEGORY_MAP.values()) - set(allowed))
check("A6  every ABO category is T2V-eligible", not missing_t2v,
      f"missing: {missing_t2v}" if missing_t2v else f"{len(allowed)} allowed")
check("P1  no ABO category requires colour",
      not (set(CATEGORY_MAP.values())
           & set(schema.attributes["color"].get("required_for_categories", []))),
      "measured coverage is 45-93%; requiring it repeats H10")

width = max(len(n) for n, _, _ in checks)
for name, ok, detail in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {name:<{width}}  {detail}")

failed = [n for n, ok, _ in checks if not ok]
assert not failed, f"\n\nWRONG BRANCH OR STALE INSTALL. Missing: {failed}"
print("\n✅ all fixes present — safe to proceed")

## Step 2 · Test suite (162)

In [ ]:
!python -m pytest -q 2>&1 | tail -5

## Step 3 · Pin γ and record it with the run

γ has had four different values across the repo (0.40 in the config, 0.60 in
the docs and the ABO analysis, 0.85 in `paper_concepts.md`, 0.448 recalibrated).
Nobody can say which produced which published number — that is finding C1, and
it happened because recalibration edits `configs/tiger.yaml` in place with no
record.

Set it **once**, here, and write the effective values into the outputs so this
run is self-documenting.

`GAMMA = 0.60` reproduces the operating point the Fashion tables were built at
(the committed confidence histogram is captioned "Gamma threshold (0.6)").
Change it only deliberately.

In [ ]:
import json, re, yaml
from pathlib import Path

GAMMA = 0.60          # operating point for THIS run
NOISE_SEED = 7
BASELINE_SEED = 42    # seeds the random-routing baseline (A3b)

cfg_path = Path("configs/tiger.yaml")
text = cfg_path.read_text()

# Surgical line edit, NOT yaml.safe_dump: dumping would round-trip the file and
# silently delete every comment in it, including the rationale for the T2V
# allowlist and the noise rates. Config comments are the only place several of
# these decisions are recorded.
# Replace the whole line including its trailing comment. Leaving the old
# comment behind ("lowered from 0.60 ...") next to a value of 0.60 is precisely
# the config/doc drift that produced finding C1 in the first place.
text, n = re.subn(
    r"^(\s*)gamma:.*$",
    rf"\g<1>gamma: {GAMMA}  # Eq. 22 confidence gate; pinned by tiger_corrected_run.ipynb",
    text, count=1, flags=re.M)
assert n == 1, "could not find `gamma:` in configs/tiger.yaml"

if not re.search(r"^eval:", text, flags=re.M):
    text += ("\neval:\n"
             "  # Seeds the random-routing baseline so the 'No Arbiter' row is\n"
             "  # reproducible (finding A3b).\n"
             f"  random_baseline_seed: {BASELINE_SEED}\n")
else:
    text, n = re.subn(r"^(\s*random_baseline_seed:\s*)\d+", rf"\g<1>{BASELINE_SEED}",
                      text, count=1, flags=re.M)
    assert n == 1
cfg_path.write_text(text)

cfg = yaml.safe_load(cfg_path.read_text())
assert cfg["arbiter"]["gamma"] == GAMMA
assert cfg["eval"]["random_baseline_seed"] == BASELINE_SEED

manifest = {
    "branch": "docs/fixes-backlog-audit",
    "gamma": GAMMA,
    "noise_seed": NOISE_SEED,
    "random_baseline_seed": BASELINE_SEED,
    "dismiss_threshold": cfg["arbiter"]["dismiss_threshold"],
    "t2v_allowed_categories": cfg["arbiter"]["t2v_policy"]["allowed_categories"],
    "precision_floor": cfg["sieve"]["precision_floor"],
    "clip_model": cfg["models"]["clip_model_name"],
    "independent_verifier": cfg["models"]["independent_verifier"],
    "fusion_applied": False,
}
Path("data/outputs").mkdir(parents=True, exist_ok=True)
Path("data/outputs/run_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))


---
# Phase A · Synthetic catalogue → detection numbers

Regenerates the detection table (precision / recall / F1 and per-error-type
recall). `synthgen` has been unbuildable since the ABO schema edit — D5 is what
makes this step run at all.

In [ ]:
!python -m tiger.cli synthgen

In [ ]:
!python -m tiger.cli calibrate

**`train-arbiter` is the slow one** — it runs eight noise seeds end to end. Expect several minutes.

In [ ]:
!python -m tiger.cli train-arbiter

In [ ]:
!python -m tiger.cli calibrate-fusion

### Detection sweep and ablations

`calibrate-fusion` output above is worth reading closely: A8 changed how
per-signal precision is scored, so a signal that previously cleared the 0.85
floor may now be quarantined. That is the correction working.

In [ ]:
!python -m tiger.cli sweep --seeds 7,8,9,10,11

In [ ]:
!python -m tiger.cli ablate --seeds 7,8,9,10,11

## Results

Two columns are new (A5). `Attr Accuracy` covers colour, material and pattern
instead of colour alone; `T2V Accuracy` scores image repairs, which were
previously not scored at all. Every metric now carries its own N — the old
headline was 52.6% on n=19 and the denominator appeared only in a LaTeX caption.

In [ ]:
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 50)

df = pd.read_csv("data/outputs/repair_ablations_summary.csv")
cols = [c for c in ["Configuration", "Repaired", "Escalated", "Total Attempted",
                    "Attr Accuracy", "Attr Cases", "T2V Accuracy", "T2V Cases",
                    "Color Accuracy", "V2T Cases"] if c in df.columns]
display(df[cols])

full = df[df["Configuration"] == "Full System"]
nog  = df[df["Configuration"].str.contains("Gamma", na=False)]
if len(full) and len(nog):
    same = (full["Repaired"].iat[0] == nog["Repaired"].iat[0]
            and full["Escalated"].iat[0] == nog["Escalated"].iat[0])
    print("\nGamma gate row vs Full System:",
          "STILL IDENTICAL — investigate, A1 should have separated these" if same
          else "different ✅ — the gate is now genuinely ablated")

### V2T estimator attribution

Every colour repair records which estimator supplied the value (the HSV pixel
histogram or the CLIP probe) and whether the *other* one would have been right.

This report decides what to work on next: if `always pixel` far exceeds
`always probe`, the encoder is the bottleneck (B7); if the reverse, the colour
estimator is (B1–B5, currently blocked on having this data locally). The
`agree / disagree` split sizes what an abstention gate would buy — that is the
parked ⚑ item B6.

In [ ]:
import pandas as pd
from pathlib import Path

p = Path("data/outputs/v2t_estimator_diagnostics.csv")
if p.exists():
    d = pd.read_csv(p)
    display(d.head(30))
    print("\nrows:", len(d))
else:
    print("No scored V2T cases in this run — nothing to attribute.")

## Export everything

In [ ]:
!cp -r data/outputs data/thresholds data/processed /kaggle/working/ 2>/dev/null
!cd /kaggle/working && zip -rq tiger_corrected_run.zip outputs thresholds processed phaseA_synthetic
!ls -la /kaggle/working/tiger_corrected_run.zip
print("\n✅ Download tiger_corrected_run.zip from the right sidebar.")

---
## What to do with these numbers

1. **Compare against `paper_assets/paper_draft_materials.md` §1.** Expect movement.
   Four independent defects fed that table; none of them biased it in a
   predictable direction, so do not assume the corrected numbers are worse.

2. **§7.5 and `ROADMAP_PROGRESS.md` H11 must be rewritten or withdrawn** (E3).
   They explain why Full System and No-Gamma-Gate matched. They matched because
   A1 made them the same run. If the corrected run still shows overlap, prove it
   properly by intersecting the escalated `row_id` sets — identical totals are
   not identical sets.

3. **`reviewer_defense.md` Attack 3 is the urgent one** (E2). It presents the
   47.4% as "safely escalated to a human". They were not: that figure is error
   among rows the system *repaired and committed*. Escalations are the separate
   269. Fix the claim before anyone reads it.

4. **Report N with every accuracy figure.** The CSV now carries them.

5. **Then run the ABO notebook** on this branch. A6 opened the T2V path that was
   closed for the entire cross-domain evaluation, so RQ3's image-repair evidence
   does not exist yet.

Still open and needing you specifically:
- **A2** — pin a Gemini model ID with a live key (`genai.list_models()`), and
  determine whether any published number came from a `--vlm-judge` run.
- **⚑ B6, ⚑ D4** — parked design decisions, not bugs.

## Results

`sweep` gives pooled detection metrics with product-level bootstrap CIs;
`ablate` gives the per-configuration breakdown.

Two things to check against the previous numbers:

* **`full` should still be ≈ P 0.793 / R 0.924 / F1 0.853.** It is unaffected by
  A7 and A8, so a large move means something else changed.
* **`full_fusion` and the per-signal precision table will move**, because A8
  changed what counts as a hit. A probe that previously cleared the floor may
  now be quarantined — that is the correction working, not a regression.

In [ ]:
import json
from pathlib import Path

sweep = json.loads(Path("data/outputs/detection_metrics_sweep.json").read_text())
p = sweep["pooled"]
print(f"pooled  P={p['precision']:.3f}  R={p['recall']:.3f}  F1={p['f1']:.3f}"
      f"   (n={p['n_products']} products, {p['n_rows']} rows)")
ci = p.get("bootstrap95_product_level", {})
if ci:
    print(f"  95% CI  P {ci['precision'][0]:.3f}-{ci['precision'][1]:.3f}"
          f"  R {ci['recall'][0]:.3f}-{ci['recall'][1]:.3f}")

print("\nper-error-type recall:")
for lab, v in sorted(p["recall_by_label"].items()):
    print(f"  {lab:<16s} {v['recall']:.3f}  ({v['caught']}/{v['total']})")

abl = json.loads(Path("data/outputs/ablations.json").read_text())
print("\nablations:")
for name in ("random@budget", "text_only", "global_only", "probes_only",
             "no_loo", "full", "full_fusion"):
    if name in abl:
        r = abl[name]
        print(f"  {name:<16s} P={r['precision']:.3f} R={r['recall']:.3f} F1={r['f1']:.3f}")

g, f = abl.get("global_only", {}), abl.get("full", {})
if g and f:
    print(f"\nmutate_text recall: CLIP-only {g['recall_by_label']['mutate_text']['recall']:.3f}"
          f"  ->  with probes {f['recall_by_label']['mutate_text']['recall']:.3f}")
    print("  (this is the paper's strongest verified result -- E9 notes it is")
    print("   currently mis-credited to LOO masking, which contributes nothing here)")

## Export

In [ ]:
!cp -r data/outputs data/thresholds data/processed /kaggle/working/ 2>/dev/null
!cd /kaggle/working && zip -rq tiger_detection_run.zip outputs thresholds processed
!ls -la /kaggle/working/tiger_detection_run.zip
print("\n✅ Download tiger_detection_run.zip from the right sidebar.")

---
## Next

Run **`tiger_abo_corrected_run.ipynb`** for the repair-side numbers on the two
official-ABO verticals. Together they replace every table in `paper_assets/`.

Then `E9`: the ablation labels the contrastive probes "LOO Probes Only" and
"No LOO Masking". Eq. 18 leave-one-out lives in the Analyzer and runs only on
already-flagged rows — it contributes nothing to detection. Correcting this
runs in your favour: the probe result is stronger and more novel than the LOO
story the paper currently tells.